In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
data1 = pd.read_csv("/kaggle/input/dataset/data_csv.csv")
data2 = pd.read_csv("/kaggle/input/autism-screening-for-toddlers/Toddler Autism dataset July 2018.csv")

df1 = pd.concat([data1.iloc[:,1:11], data1.iloc[:,[12,22,23,24,25,26,27]]], axis=1)
df2 = pd.concat([data2.iloc[:,1:12], data2.iloc[:,13:]], axis=1)

df2['Age_Mons'] = (df2['Age_Mons']/12).astype(int)
df2.columns = df1.columns

data_fin = pd.concat([df2, df1], axis=0)
data_fin['Sex'] = data_fin['Sex'].replace({'f':'F','m':'M'})
data_fin['Jaundice'] = data_fin['Jaundice'].replace({'yes':'Yes','no':'No'})
data_fin['Family_mem_with_ASD'] = data_fin['Family_mem_with_ASD'].replace({'yes':'Yes','no':'No'})

data_fin.fillna(data_fin.mode().iloc[0], inplace=True)

le = LabelEncoder()
for col in data_fin.select_dtypes('object').columns:
    data_fin[col] = le.fit_transform(data_fin[col])
X = data_fin.drop(columns=["ASD_traits"])
y = data_fin["ASD_traits"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.23, random_state=45, stratify=y
)

sc = MinMaxScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

params = {
    'max_depth': [5,10,20],
    'min_samples_leaf': [5,10,20],
    'n_estimators': [50,100,200]
}

grid = GridSearchCV(
    rf, params, cv=4, scoring="accuracy", n_jobs=-1, verbose=1
)

grid.fit(X_train_scaled, y_train)

rf_best = grid.best_estimator_

p_tab = rf_best.predict_proba(X_test_scaled)[:,1]

print("Tabular Accuracy:",
      accuracy_score(y_test, (p_tab>0.5).astype(int)))


2026-01-21 21:31:31.140664: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769031091.322335      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769031091.381742      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769031091.811763      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769031091.811831      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769031091.811834      24 computation_placer.cc:177] computation placer alr

Fitting 4 folds for each of 27 candidates, totalling 108 fits
Tabular Accuracy: 0.9484978540772532


In [2]:
DATASET_DIR = "/kaggle/input/autism-image-dataset/autism_images"
AUT_DIR = os.path.join(DATASET_DIR, "autistic")
NONAUT_DIR = os.path.join(DATASET_DIR, "non_autistic")

IMG_SIZE = (224,224)

def load_images(folder_list):
    X, y = [], []
    for label, folder in enumerate(folder_list):
        for fname in os.listdir(folder):
            try:
                path = os.path.join(folder, fname)
                img = tf.keras.preprocessing.image.load_img(path, target_size=IMG_SIZE)
                img = tf.keras.preprocessing.image.img_to_array(img)
                X.append(img)
                y.append(label)
            except:
                pass
    return np.array(X, dtype="float32"), np.array(y)

X_img, y_img = load_images([AUT_DIR, NONAUT_DIR])
X_img = X_img / 255.0


In [3]:
X_train_img, X_val_img, y_train_img, y_val_img = train_test_split(
    X_img, y_img, test_size=0.2, stratify=y_img, random_state=42
)
model_cnn = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32, (7,7), activation='relu', padding='same'),
    layers.Conv2D(32, (7,7), activation='relu', padding='same'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (5,5), activation='relu', padding='same'),
    layers.Conv2D(64, (5,5), activation='relu', padding='same'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(1024, activation='relu'),
    layers.Dense(512, activation='relu'),

    layers.Dense(1, activation='sigmoid')
])

model_cnn.compile(
    optimizer=optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


I0000 00:00:1769031144.278475      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [4]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=10, restore_best_weights=True
    )
]

model_cnn.fit(
    X_train_img, y_train_img,
    validation_data=(X_val_img, y_val_img),
    epochs=80,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)
p_img = model_cnn.predict(X_val_img).reshape(-1)

print("Image Accuracy:",
      accuracy_score(y_val_img, (p_img>0.5).astype(int)))


Epoch 1/80


I0000 00:00:1769031151.354326    1612 service.cc:152] XLA service 0x7d21340141f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769031151.354367    1612 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1769031151.915255    1612 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-01-21 21:32:36.774642: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng2{k2=3,k3=0} for conv %cudnn-conv-bw-input.9 = (f32[16,32,224,224]{3,2,1,0}, u8[0]{0}) custom-call(f32[16,32,224,224]{3,2,1,0} %bitcast.4324, f32[32,32,7,7]{3,2,1,0} %bitcast.3519), window={size=7x7 pad=3_3x3_3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", metadata={op_type="Conv2DBackpropInput" op_name="gradient_tape/sequential_1/conv2d_1_2/convolution/Conv2DBackpropInput" source_file="/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/ops.py" source_line=12

  2/147 ━━━━━━━━━━━━━━━━━━━━ 10s 71ms/step - accuracy: 0.6094 - loss: 0.6906 

I0000 00:00:1769031163.895970    1612 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


146/147 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.5386 - loss: 0.6958

2026-01-21 21:32:57.575190: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng2{k2=3,k3=0} for conv %cudnn-conv-bw-input.9 = (f32[12,32,224,224]{3,2,1,0}, u8[0]{0}) custom-call(f32[12,32,224,224]{3,2,1,0} %bitcast.4324, f32[32,32,7,7]{3,2,1,0} %bitcast.3519), window={size=7x7 pad=3_3x3_3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", metadata={op_type="Conv2DBackpropInput" op_name="gradient_tape/sequential_1/conv2d_1_2/convolution/Conv2DBackpropInput" source_file="/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2026-01-21 21:32:57.597027: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.022050649s
Trying a

147/147 ━━━━━━━━━━━━━━━━━━━━ 41s 176ms/step - accuracy: 0.5391 - loss: 0.6955 - val_accuracy: 0.5867 - val_loss: 0.6764
Epoch 2/80
147/147 ━━━━━━━━━━━━━━━━━━━━ 11s 77ms/step - accuracy: 0.6885 - loss: 0.5884 - val_accuracy: 0.6446 - val_loss: 0.6470
Epoch 3/80
147/147 ━━━━━━━━━━━━━━━━━━━━ 11s 77ms/step - accuracy: 0.7373 - loss: 0.5225 - val_accuracy: 0.6888 - val_loss: 0.5776
Epoch 4/80
147/147 ━━━━━━━━━━━━━━━━━━━━ 11s 73ms/step - accuracy: 0.7767 - loss: 0.4563 - val_accuracy: 0.6769 - val_loss: 0.6549
Epoch 5/80
147/147 ━━━━━━━━━━━━━━━━━━━━ 11s 77ms/step - accuracy: 0.8275 - loss: 0.3664 - val_accuracy: 0.7330 - val_loss: 0.5810
Epoch 6/80
147/147 ━━━━━━━━━━━━━━━━━━━━ 11s 73ms/step - accuracy: 0.8646 - loss: 0.3060 - val_accuracy: 0.7058 - val_loss: 0.7437
Epoch 7/80
147/147 ━━━━━━━━━━━━━━━━━━━━ 11s 73ms/step - accuracy: 0.9420 - loss: 0.1690 - val_accuracy: 0.6956 - val_loss: 0.8634
Epoch 8/80
147/147 ━━━━━━━━━━━━━━━━━━━━ 11s 73ms/step - accuracy: 0.9761 - loss: 0.0866 - val_accura

In [5]:
n = min(len(p_tab), len(p_img))

p_tab_f = p_tab[:n]
p_img_f = p_img[:n]
y_f = y_test.values[:n]   # stronger labels


In [6]:
p_final = 0.7 * p_tab_f + 0.3 * p_img_f

y_pred = (p_final > 0.5).astype(int)

print("🔥 Multimodal Accuracy:",
      accuracy_score(y_f, y_pred))

print(classification_report(y_f, y_pred))
print(confusion_matrix(y_f, y_pred))


🔥 Multimodal Accuracy: 0.9455782312925171
              precision    recall  f1-score   support

           0       0.94      0.92      0.93       236
           1       0.95      0.96      0.95       352

    accuracy                           0.95       588
   macro avg       0.95      0.94      0.94       588
weighted avg       0.95      0.95      0.95       588

[[217  19]
 [ 13 339]]
